# 🎓 Transfer Learning in TensorFlow 

---

## 🎯 What We Are Doing Today:
1. **Why Transfer Learning?** Why train a model from scratch for weeks when we can use a pre-trained model (like **ResNet** or **EfficientNet**) trained on 1.4 million images?
2. **Step 1: Feature Extraction** — Use the pre-trained model **as is** (frozen) and train only our new output layer.
3. **Step 2: Fine-Tuning** — Unfreeze the top layers and train with a very small learning rate to boost accuracy even higher.
4. **Step 3: Student Task** — The Ultimate CIFAR-100 Shootout & Unfreezing Challenge!


## 1. Import Libraries
Let's import TensorFlow, Keras, Matplotlib, NumPy, and Pathlib.


In [ ]:
import pathlib
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers

print("TensorFlow Version:", tf.__version__)


## 2. Load the Dataset (The Standard Flowers Dataset)

We download the famous **Flowers Dataset** (3,670 images across 5 classes: `daisy`, `dandelion`, `roses`, `sunflowers`, `tulips`).

We use `tf.keras.utils.image_dataset_from_directory`, which automatically:
- Loads images from the 5 flower subfolders
- Resizes them to `(224, 224)`
- Splits them into **80% Training** and **20% Validation**
- Converts labels to integers (0, 1, 2, 3, 4)


In [ ]:
# Step 1: Download and extract flowers dataset
dataset_url = "https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz"
data_dir = tf.keras.utils.get_file('flower_photos', origin=dataset_url, untar=True)
data_dir = pathlib.Path(data_dir)

# Step inside the extracted subfolder if needed so it grabs the 5 flower classes
if (data_dir / 'flower_photos').exists():
    data_dir = data_dir / 'flower_photos'

# Image & Batch settings
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

# Step 2: Load Training Set (80%)
train_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

# Step 3: Load Validation Set (20%)
val_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

# Extract class names (5 classes: daisy, dandelion, roses, sunflowers, tulips)
class_names = train_ds.class_names
print("\n🌸 Detected Classes:", class_names)
print(f"Total Classes: {len(class_names)}")


## 3. Visualize the Data
Let's plot 9 images from our dataset with their true flower class names.


In [ ]:
plt.figure(figsize=(10, 10))

# Take one batch from the training dataset
for images, labels in train_ds.take(1):
    for i in range(9):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(class_names[labels[i]])
        plt.axis("off")

plt.show()


## 4. Performance Optimization (`cache` & `prefetch`)
This keeps images in RAM after the first epoch and prepares the next batch in the background while the GPU trains.


In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)


## 5. Data Augmentation
To prevent overfitting, we randomly flip and rotate images during training.  
These Keras layers run on GPU and are **active ONLY during training** (automatically turned off during testing).


In [ ]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.2),
    layers.RandomZoom(0.1),
])


## 6. Step 1: Feature Extraction (Using Pretrained Model AS-IS)

### How It Works:
1. **Load Pre-trained Backbone**: We load **`EfficientNetB0`** pre-trained on ImageNet (1.4 million images).
   - `include_top=False`: Removes the original 1,000-class ImageNet classification head.
2. **FREEZE the Backbone**: `base_model.trainable = False`.
   - We use the pre-trained model **as-is**! All feature extraction weights are frozen.
3. **Add Our New Head**:
   - `GlobalAveragePooling2D()`: Converts spatial feature maps $(7 \times 7 \times 1280)$ into a 1D vector $(1280)$.
   - `Dropout(0.2)`: Regularization to prevent overfitting.
   - `Dense(5)`: Outputs prediction logits for our 5 flower classes.

```
[ Input Image (224x224x3) ]
            │
            ▼
   [ Data Augmentation ]
            │
            ▼
[ Pretrained Base Model ]  ──► FROZEN (trainable = False) (Used AS-IS)
            │
            ▼
[ GlobalAveragePooling2D ]
            │
            ▼
      [ Dropout(0.2) ]
            │
            ▼
   [ Dense(5 Classes) ]    ──► TRAINABLE (Our new 5-class head)
```


In [ ]:
# 1. Load Pretrained Backbone (EfficientNetB0)
base_model = tf.keras.applications.EfficientNetB0(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)

# 2. FREEZE the base model (use as-is)
base_model.trainable = False

# 3. Build the full model with our custom 5-class head
model = tf.keras.Sequential([
    layers.Input(shape=(224, 224, 3)),
    data_augmentation,
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.2),
    layers.Dense(len(class_names))  # 5 output classes
])

# View summary (Notice: Only ~6,400 trainable params!)
model.summary()


### Compile and Train Phase 1 (Feature Extraction)
We compile with standard `'adam'` and train for 5 epochs. Notice how accuracy quickly climbs to ~85-90%!


In [ ]:
# Compile the model
model.compile(
    optimizer='adam',
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)

# Train for 5 epochs
EPOCHS_PHASE_1 = 5
print("🚀 Training Feature Extractor (Phase 1)...")
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_PHASE_1
)


## 7. Step 2: Fine-Tuning (Selective Unfreezing)

### Why Fine-Tune?
Now that our new top classification layer is trained and stable, we can optionally unfreeze the top layers of the base model so they adapt slightly to flower features.

### The 2 Golden Rules of Fine-Tuning:
1. **Unfreeze only top layers** (keep low-level edge/texture layers frozen).
2. **Use a very small learning rate** (e.g., `1e-5`, which is 100x smaller than standard `1e-3`). A large learning rate would destroy pre-trained features (*catastrophic forgetting*)!


In [ ]:
# Step 1: Unfreeze the base model
base_model.trainable = True

# Step 2: Freeze all layers EXCEPT the last 20 layers
print(f"Total layers in base model: {len(base_model.layers)}")
for layer in base_model.layers[:-20]:
    layer.trainable = False

# Step 3: Recompile with a 100x smaller learning rate (1e-5)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)

print("✅ Model recompiled with learning_rate = 1e-5 for fine-tuning!")


In [ ]:
# Step 4: Fine-tune for 5 more epochs (total 10 epochs)
EPOCHS_FINE_TUNE = 5
TOTAL_EPOCHS = EPOCHS_PHASE_1 + EPOCHS_FINE_TUNE

print(f"🚀 Fine-Tuning from epoch {EPOCHS_PHASE_1} to {TOTAL_EPOCHS}...")
history_fine = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=TOTAL_EPOCHS,
    initial_epoch=history.epoch[-1] + 1
)


## 8. Visualize Accuracy & Loss Trajectory
Let's plot the training and validation curves across both Phase 1 (Feature Extraction) and Phase 2 (Fine-Tuning).


In [ ]:
# Combine metrics from both phases
acc = history.history['accuracy'] + history_fine.history['accuracy']
val_acc = history.history['val_accuracy'] + history_fine.history['val_accuracy']

loss = history.history['loss'] + history_fine.history['loss']
val_loss = history.history['val_loss'] + history_fine.history['val_loss']

plt.figure(figsize=(14, 5))

# Plot Accuracy
plt.subplot(1, 2, 1)
plt.plot(acc, label='Training Accuracy', marker='o')
plt.plot(val_acc, label='Validation Accuracy', marker='s')
plt.axvline(x=EPOCHS_PHASE_1 - 0.5, color='red', linestyle='--', label='Start Fine-Tuning')
plt.title('Training & Validation Accuracy', fontsize=12, fontweight='bold')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)

# Plot Loss
plt.subplot(1, 2, 2)
plt.plot(loss, label='Training Loss', marker='o')
plt.plot(val_loss, label='Validation Loss', marker='s')
plt.axvline(x=EPOCHS_PHASE_1 - 0.5, color='red', linestyle='--', label='Start Fine-Tuning')
plt.title('Training & Validation Loss', fontsize=12, fontweight='bold')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend(loc='upper right')
plt.grid(True, alpha=0.3)

plt.show()


### Visual Predictions on Validation Images
Let's see our model predict on new images from the validation set!


In [ ]:
plt.figure(figsize=(12, 10))

# Take one batch of images from validation set
for images, labels in val_ds.take(1):
    predictions = model.predict(images)
    pred_labels = np.argmax(predictions, axis=1)
    
    for i in range(9):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        
        true_name = class_names[labels[i]]
        pred_name = class_names[pred_labels[i]]
        
        is_correct = (true_name == pred_name)
        title_color = "green" if is_correct else "red"
        
        plt.title(f"Pred: {pred_name}\nTrue: {true_name}", color=title_color, fontweight='bold')
        plt.axis("off")

plt.tight_layout()
plt.show()


---
## 📚 9. Transfer Learning Cheatsheet & Summary

| Concept | What It Does | Key Rule |
| :--- | :--- | :--- |
| **Feature Extraction** | Freezes the base model, trains only the new output head. | Base model `trainable = False`. Use standard learning rate (`1e-3`). |
| **Fine-Tuning** | Unfreezes top layers of the base model for slight weight adaptation. | Base model top layers `trainable = True`. **Must use small learning rate (`1e-5`)**! |
| **`GlobalAveragePooling2D`** | Flattens spatial $(7 \times 7 \times C)$ to $(C)$ without adding millions of parameters. | Always prefer over `Flatten()` in transfer learning! |
| **Data Augmentation** | Prevents overfitting by adding random flips/rotations on GPU. | Keras augmentation layers are active **only** during `fit()`. |
| **Preprocessing** | Normalizes pixel values. | `EfficientNet`: Built-in $[0, 255]$. <br>`ResNet50V2`/`MobileNetV2`: Needs `Rescaling(1./127.5, offset=-1)`. |

---
*Created for NTI Deep Learning & Computer Vision Lecture Series.*


# Student Task: The CIFAR-100 Architecture Shootout & Unfreezing Challenge

**Your Mission:**
You will apply Transfer Learning to the CIFAR-100 dataset (100 categories of objects).

**Deadline:**
Thursday the 3rd of the month at 11:59 PM

---

**Step-by-Step Instructions:**

**Part 1: The Model Shootout (Feature Extraction)**

* **Model A:** Build a Transfer Learning model using `ResNet50V2` as a frozen backbone.
* **Model B:** Build a Transfer Learning model using `MobileNetV2` as a frozen backbone.
* Train both on CIFAR-100 for **5 epochs** each using Feature Extraction (`base_model.trainable = False`).
* **Compare:** Which model achieved higher validation accuracy? Which one trained faster?

**Part 2: The Winner's Circle (Percentage-Based Unfreezing)**
Take the winning model from Part 1, set `base_model.trainable = True`, and test two unfreezing depths based on the total number of layers:

* **Strategy A (Top 10% Unfreezing):**
* If `ResNet50V2` wins (190 total layers): Unfreeze the last 19 layers.
* If `MobileNetV2` wins (154 total layers): Unfreeze the last 15 layers.


* **Strategy B (Deep 35% Unfreezing):**
* If `` wins (190 total layers): Unfreeze the last 65 layers.
* If `MobileNetV2` wins (154 total layers): Unfreeze the last 54 layers.



**How to implement the percentage split dynamically in Python:**

```python
# 1. Unfreeze the base model
base_model.trainable = True

# 2. Calculate the split index for your target unfreeze percentage (e.g., 0.10 or 0.35)
total_layers = len(base_model.layers)
freeze_until = int(total_layers * (1.0 - unfreeze_percentage))

# 3. Freeze all layers before the split point
for layer in base_model.layers[:freeze_until]:
    layer.trainable = False

```

* Fine-tune each strategy for **5 epochs** with `learning_rate = 1e-5`.
* **Analyze:** Which unfreezing depth gave the best final accuracy? Did deep unfreezing cause any overfitting?

---

**Helper Notes for CIFAR-100:**

* CIFAR-100 images are 32x32. Pre-trained backbones expect larger images, so add a `layers.Resizing(128, 128)` layer.
* `ResNet50V2` and `MobileNetV2` expect inputs in the range [-1, 1], so add `layers.Rescaling(1./127.5, offset=-1)`.
* Keep Batch Normalization running in inference mode during fine-tuning by calling `base_model(inputs, training=False)`.